# Receding-horizon replanning (task MPC-1)

The rule behind the console suggestion: when a what-if trigger shrinks the time left, re-check the
rest of the plan and suggest the smallest change that still gets the astronaut home with the reserve
intact. Triggers are **simulated**, not measured:

- **Suit leak:** the consumables budget drops by `leak_min`, so the deadline `budget - leak - reserve` moves earlier.
- **Dust / low visibility:** walking slows by `speed_factor`. Wind is modelled this way, as dust and
  visibility, not as a force: at Jezero's air density (about 0.018 kg/m³ from the sol 1215 MEDA
  pressure and temperature) a 20 m/s wind pushes like a 2–3 m/s breeze on Earth.

Rule, in order (code in `replanning.py`, tests in `test_replanning.py`):

1. Plan still fits: no suggestion.
2. Shorten the remaining dwells, all equally, not below `min_dwell_min`.
3. Drop one remaining stop (try each, keep the order of the rest), then two, and so on. Most stops
   kept wins, then the longest dwell, then the larger margin.
4. Nothing fits: go home now.

Never reorder stops; never drop the stop the astronaut is walking to (the planner routes between
stops only). Re-check every `replan_interval_min`, on arrival at each stop, and when a trigger changes.

Output: `out/replanning_cases.json` (inputs and expected decisions; the web port's tests use it) and
`out/replanning.png`.

In [1]:
import json, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import replanning as rp

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "notebooks" / "out"
aoi = json.loads((ROOT / "contracts" / "aoi.json").read_text())
P = rp.load_params(ROOT / "model" / "params.json")
print(P)

PLANNER = "https://cachy-nuri.tail1225de.ts.net:10000/planner/route"   # Nuri's laptop, DECISIONS.md
REFETCH = False   # True: ask the live planner again and overwrite out/route.json
if REFETCH:
    data = rp.fetch_route(PLANNER, [s["id"] for s in aoi["stops"]])
    (OUT / "route.json").write_text(json.dumps(data, indent=1))
data = json.loads((OUT / "route.json").read_text())
R, W, H = data["route"], data["walk_min"], data["home_min"]
print("route fetched", data["fetched_at"], "totals", R["totals"])

{'budget_min': 360, 'reserve_min': 60, 'dwell_min': 20, 'min_dwell_min': 10, 'replan_interval_min': 5}
route fetched 2026-11-14T16:06:05Z totals {'distance_m': 2892.11, 'walk_min': 119.65, 'dwell_min': 100.0, 'eva_min': 219.65}


## Inputs check

`cost_to_home_2m.f32` from the asset host (saved to `data/`, git-ignored). Its shape must match
`contracts/aoi.json`, and at each stop it must equal the planner's walk home: the "go home now"
option uses the raster, the other options use the planner's legs.

In [2]:
px = aoi["rasters"]["cost_to_home"]["px_m"]
w, h = int(aoi["size_m"][0] / px), int(aoi["size_m"][1] / px)
raw = np.fromfile(ROOT / "data" / "cost_to_home_2m.f32", "<f4")
assert raw.size == w * h, f"cost_to_home has {raw.size} cells, aoi.json says {w} x {h}"
cth = raw.reshape(h, w)

def cth_at(x, y):
    return float(cth[int((aoi["size_m"][1] - y) // px), int(x // px)])

for s in aoi["stops"]:
    print(f"{s['id']:16s} raster {cth_at(*s['xy']):6.2f}  planner {H[s['id']]:6.2f} min")
    assert abs(cth_at(*s["xy"]) - H[s["id"]]) < 0.5

for a in rp.timeline(R):
    print(f"arrive {a['stop']:16s} t = {a['arrive_min']:6.1f} min")
print("plan ends", R["totals"]["eva_min"], "min; deadline", P["budget_min"] - P["reserve_min"], "min")

mount-washburn   raster  22.64  planner  22.64 min
tuff-cliff       raster  26.15  planner  26.15 min
bright-angel     raster  42.80  planner  42.80 min
cheyava-falls    raster  43.77  planner  43.77 min
masonic-temple   raster  47.23  planner  47.23 min
arrive mount-washburn   t =   22.6 min
arrive tuff-cliff       t =   47.5 min
arrive bright-angel     t =   88.2 min
arrive cheyava-falls    t =  115.2 min
arrive masonic-temple   t =  152.4 min
plan ends 219.65 min; deadline 300 min


## Scenarios

In [3]:
def home_on_leg(i, frac):
    return cth_at(*rp.point_along(R["legs"][i]["path"], frac))

cases = rp.cases(data, P, home_on_leg)
for c in cases:
    e = c["expected"]
    print(f"{c['id']:28s} {e['action']:15s} dwell {e['dwell_min']:4.0f}  dropped {e['dropped']}  "
          f"margin {e['margin_min']:6.1f}  reserve at home {e['reserve_at_home_min']:5.1f}")

doc = {
    "_comment": "Receding-horizon replanning cases (task MPC-1). SYNTHETIC triggers on the real route. "
                "decide(input, walk_min, home_min, params) must return `expected`; see notebooks/replanning.py "
                "and docs/model.md, Replanning. Margin is minutes above the reserve; negative = the reserve is in use.",
    "provenance": "Simulation",
    "route_fetched_at": data["fetched_at"],
    "params": P,
    "presets": rp.PRESETS,
    "stops": data["stops"],
    "walk_min": W,
    "home_min": H,
    "cases": cases,
}
(OUT / "replanning_cases.json").write_text(json.dumps(doc, indent=1, ensure_ascii=False) + "\n")
print(len(cases), "cases written")

nominal                      none            dwell   20  dropped []  margin   80.4  reserve at home 140.4
dust-0.5-at-mount-washburn   shorten_dwells  dwell   16  dropped []  margin    3.3  reserve at home  63.3
dust-0.4-at-mount-washburn   drop_stops      dwell   20  dropped ['masonic-temple']  margin    6.5  reserve at home  66.5
dust-0.5-mid-leg             shorten_dwells  dwell   19  dropped []  margin    1.5  reserve at home  61.5
leak-90-at-tuff-cliff        shorten_dwells  dwell   17  dropped []  margin    2.4  reserve at home  62.4
leak-120-at-bright-angel     drop_stops      dwell   20  dropped ['masonic-temple']  margin    1.0  reserve at home  61.0
leak-120-during-dwell        drop_stops      dwell   20  dropped ['masonic-temple']  margin    1.0  reserve at home  61.0
leak-150-at-cheyava-falls    go_home_now     dwell    0  dropped ['masonic-temple']  margin   -9.0  reserve at home  51.0
dust-and-leak                shorten_dwells  dwell   11  dropped []  margin    0.7  rese

## Figure for the project page

Margin above the reserve over the walk: the plan as made (80 min), what happens at the trigger if
nothing changes, and with the suggestion followed. Below zero the astronaut would eat into the reserve.

In [4]:
def no_change(s):
    """End time if the plan continues unchanged after the trigger."""
    w = rp.plan_time(s["start"], s["remaining"], s.get("lead_min", 0), W, H) / s.get("speed_factor", 1)
    n = 1 + len(s["remaining"])
    end = s["t_min"] + w + max(0, P["dwell_min"] - s.get("dwell_done_min", 0)) + (n - 1) * P["dwell_min"]
    return P["budget_min"] - s.get("leak_min", 0) - P["reserve_min"] - end, end

names = {s["id"]: s["name"] for s in aoi["stops"]}
show = ["dust-0.5-at-mount-washburn", "leak-120-at-bright-angel"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
nominal = P["budget_min"] - P["reserve_min"] - R["totals"]["eva_min"]
for ax, cid in zip(axes, show):
    c = next(c for c in cases if c["id"] == cid)
    s, e = c["input"], c["expected"]
    nc, nc_end = no_change(s)
    te = s["t_min"]
    ax.axhline(0, color="#a50026", lw=1)
    ax.text(335, -2, "reserve line (60 min held back)", color="#a50026", fontsize=8, va="top", ha="right")
    ax.plot([0, te], [nominal, nominal], color="#888", lw=2, label="plan as made")
    ax.plot([te, R["totals"]["eva_min"]], [nominal, nominal], color="#888", lw=1.5, ls=":")
    ax.plot([te, te, nc_end], [nominal, nc, nc], color="#d73027", lw=2, ls="--", label="trigger, no change")
    ax.plot([te, te, e["end_min"]], [nominal, e["margin_min"], e["margin_min"]], color="#1a9850", lw=2.5,
            label="trigger, suggestion followed")
    ax.axvline(te, color="#555", lw=0.8, ls=":")
    trig = (f"dust: speed x{s['speed_factor']}" if s.get("speed_factor", 1) != 1 else f"suit leak: -{s['leak_min']:.0f} min")
    if e["action"] == "shorten_dwells":
        what = f"shorten dwells to {e['dwell_min']:.0f} min"
    elif e["action"] == "drop_stops":
        what = "skip " + ", ".join(names[d] for d in e["dropped"])
    else:
        what = e["action"].replace("_", " ")
    ax.set_title(f"{trig} at {names[s['start']]} (t = {te:.0f} min)\nsuggestion: {what}", fontsize=10)
    ax.set_xlabel("elapsed time (min)")
    ax.set_xlim(0, 340)
axes[0].set_ylabel("margin above reserve (min)")
axes[1].legend(loc="center right", fontsize=8, frameon=False)
fig.text(0.01, 0.005, "SIMULATION: triggers are what-if inputs; route from the Ariadne planner on HiRISE-derived slopes.",
         fontsize=7, color="#555")
fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(OUT / "replanning.png", dpi=150)